# RAFT Training Data Exploration

This notebook provides a comprehensive exploration of the RAFT (Retrieval-Augmented Fine-Tuning). 
The primary goals are:

1. **Validate generation quality** — confirm that oracle/distractor ratios and answer formats are correct.
2. **Determine `max_new_tokens`** — by analysing the distribution of CoT answer lengths.
3. **Determine `max_seq_length`** — by analysing the distribution of full instruction (context + question) lengths.

## Dataset Overview

The RAFT dataset is stored in `data/training_data_raft` as three JSONL splits:

| Split | File | Purpose |
|-------|------|---------|
| Train | `train.jsonl` | Fine-tuning examples (80 %) |
| Validation | `validation.jsonl` | In-training evaluation / early stopping (10 %) |
| Test | `test.jsonl` | Held-out evaluation (10 %) |

Each record contains:

| Field | Description |
|-------|-------------|
| `question` | A factual question derived from a source document chunk |
| `context` | Retrieved passages — either oracle plus distractors, or distractor-only passages |
| `oracle_context` | The specific chunk containing the correct answer |
| `cot_answer` | Chain-of-thought answer with `##begin_quote##` citations and a final `<ANSWER>:` tag |
| `instruction` | Fully formatted model input: `<DOCUMENT>` blocks + question |
| `type` | `"oracle"` (answer chunk present) or `"distractor"` (answer chunk withheld) |

## Notebook Sections

1. Load Dataset  
2. Dataset Balance: Oracle vs Distractor  
3. CoT Answer Length Analysis → guides `max_new_tokens`  
4. Instruction (Context) Length Analysis → guides `max_seq_length`  
5. Persist Computed Columns

## 1. Load Dataset

Load the three JSONL splits into Pandas DataFrames and verify row counts. The 80/10/10 split is produced by `raft_datagen.py` at generation time.

In [ ]:
from pathlib import Path
import pandas as pd
import os

dataset_dir = next(
    (
        parent / "data" / "training_data_raft"
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "data" / "training_data_raft").is_dir()
    ),
    None,
)

if dataset_dir is None:
    raise FileNotFoundError(
        "Could not find data/training_data_raft from the current working directory: "
        f"{Path.cwd()}"
    )

split_paths = {
    split: dataset_dir / f"{split}.jsonl"
    for split in ("train", "test", "validation")
}

for split, path in split_paths.items():
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError(f"{split} dataset is missing or empty: {path}")

print(f"Loading dataset from: {dataset_dir.resolve()}")
train_df = pd.read_json(split_paths["train"], lines=True)
test_df = pd.read_json(split_paths["test"], lines=True)
validation_df = pd.read_json(split_paths["validation"], lines=True)
print("Number of samples: " f"train={len(train_df)}, test={len(test_df)}, validation={len(validation_df)}")
train_df.head()

## 2. Dataset Balance: Oracle vs Distractor

Each sample is labelled `"oracle"` (the correct chunk is present in the retrieved context) or `"distractor"` (the oracle chunk is withheld). Following the RAFT paper, roughly **80 %** of samples include the oracle with distractors and **20 %** contain only distractors; both are still trained toward the oracle-derived answer `A*`.

Verify that the generation script produced the expected balance across all three splits.

In [ ]:
print(train_df["type"].value_counts())
print(test_df["type"].value_counts())
print(validation_df["type"].value_counts())

# % of examples where the oracle chunk is included in the context
print("Train oracle chunk inclusion rate:", (train_df["type"] == "oracle").mean())
print("Test oracle chunk inclusion rate:", (test_df["type"] == "oracle").mean())
print("Validation oracle chunk inclusion rate:", (validation_df["type"] == "oracle").mean())

### 2.1 Sample Oracle Example

Inspect a single oracle-type record to verify the CoT answer format. A well-formed answer should:
- Cite evidence with `##begin_quote##...##end_quote##` markers drawn verbatim from the context
- End with a concise `<ANSWER>: ...` tag containing the final response

In [ ]:
sample_oracle = train_df[train_df["type"]== "oracle"].iloc[0]
print("Question:", sample_oracle["question"])
print("COT Answer:", sample_oracle["cot_answer"])

### 2.2 Sample Distractor Example

In a distractor example the oracle chunk is **absent** from the context. RAFT still trains the model to produce the oracle-derived answer `A*`, which mirrors the paper's `Q + D_1 ... D_k -> A*` examples.

In [ ]:
sample_distractor = train_df[train_df["type"]== "distractor"].iloc[0]
print("Question:", sample_distractor["question"])

print("Context sentences:\n\n")
print("="*50)
for sentence in sample_distractor["context"]["sentences"][0]:
    print("-", sentence)

print("COT Answer:", sample_distractor["cot_answer"])

## 3. CoT Answer Length Analysis

Understanding the distribution of CoT answer lengths (measured in whitespace-delimited words) is critical for setting `max_new_tokens` during fine-tuning and inference.

**Guidance:** a reasonable `max_new_tokens` should cover at least the **95th percentile** of the answer word count. Multiply by ~1.3 to convert words to sub-word tokens.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df["cot_answer_length"] = train_df["cot_answer"].apply(lambda x: len(x.split()))
test_df["cot_answer_length"] = test_df["cot_answer"].apply(lambda x: len(x.split()))
validation_df["cot_answer_length"] = validation_df["cot_answer"].apply(lambda x: len(x.split()))
plt.figure(figsize=(12, 6))
sns.histplot(train_df["cot_answer_length"], color="blue", label="Train", kde=True, stat="density")
sns.histplot(test_df["cot_answer_length"], color="orange", label="Test", kde=True, stat="density")
sns.histplot(validation_df["cot_answer_length"], color="green", label="Validation", kde=True, stat="density")
plt.title("Distribution of CoT Answer Lengths")
plt.xlabel("Number of Words in CoT Answer")
plt.legend()
plt.show()

### 3.1 Percentile Statistics

Compute descriptive statistics with key percentiles to identify a safe `max_new_tokens` cutoff.

In [ ]:
# order by length and look at percentiles for train, test, and validation sets
print("Train CoT answer length percentiles:")
print(train_df["cot_answer_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print("\nTest CoT answer length percentiles:")
print(test_df["cot_answer_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print("\nValidation CoT answer length percentiles:")
print(validation_df["cot_answer_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

## 4. Instruction (Context) Length Analysis

The `instruction` field is the full model input: multiple `<DOCUMENT>` blocks followed by the question. Depending on `type`, those blocks contain either oracle plus distractors or distractors only. Its word-count distribution determines a safe value for `max_seq_length` in the fine-tuning configuration.

> **Important:** samples longer than `max_seq_length` tokens are silently **right-truncated** by the tokeniser, which can clip the question and cause the model to learn from malformed examples.

**Guidance:** set `max_seq_length` ≥ 99th percentile of instruction word count × 1.3 (words-to-tokens expansion factor).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_df["instruction_length"] = train_df["instruction"].apply(lambda x: len(x.split()))
test_df["instruction_length"] = test_df["instruction"].apply(lambda x: len(x.split()))
validation_df["instruction_length"] = validation_df["instruction"].apply(lambda x: len(x.split()))

plt.figure(figsize=(12, 6))
sns.histplot(train_df["instruction_length"], color="blue", label="Train", kde=True, stat="density")
sns.histplot(test_df["instruction_length"], color="orange", label="Test", kde=True, stat="density")
sns.histplot(validation_df["instruction_length"], color="green", label="Validation", kde=True, stat="density")
plt.title("Distribution of Instruction Lengths")
plt.xlabel("Number of Words in Instruction")
plt.legend()
plt.show()


### 4.1 Percentile Statistics

Compute descriptive statistics to determine a safe `max_seq_length`. Pay attention to the 95th and 99th percentiles — multiply by ~1.3 to estimate sub-word token counts.

In [ ]:
print("Train instruction length percentiles:")
print(train_df["instruction_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print("\nTest instruction length percentiles:")
print(test_df["instruction_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
print("\nValidation instruction length percentiles:")
print(validation_df["instruction_length"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

# Estimate token count at the 99th percentile (words × 1.3 expansion factor)
p99_words = train_df["instruction_length"].quantile(0.99)
print(f"\nTrain 99th percentile: {p99_words:.0f} words ≈ {p99_words * 1.3:.0f} tokens")
print("Recommended max_seq_length:", int(p99_words * 1.3 * 1.1 / 128 + 1) * 128, "(rounded up to nearest 128)")


In [ ]:
MAX_NEW_TOKENS = train_df["cot_answer_length"].max() # 512
MAX_SEQ_LEN = train_df["instruction_length"].max() # around 1600

# Based on the analysis, we can set max_new_tokens and max_seq_length to accommodate the longest instructions and answers with some buffer. 
print(f"Recommended max_new_tokens: {MAX_NEW_TOKENS}")
print(f"Recommended max_seq_length: {int(MAX_SEQ_LEN * 1.3 * 1.1 / 128 + 1) * 128} (rounded up to nearest 128)")

## Summary 

Based on the analysis above, the following hyperparameters are suggested for the fine-tuning notebook (`raft-finetuning-slm.ipynb`):

| Parameter | Suggested Value | Rationale |
|-----------|-----------------|-----------|
| `max_new_tokens` | 256 – 512 | Covers ≥ 95th percentile of CoT answer word counts |
| `max_seq_length` | 896 – 1024 | Covers ≥ 99th percentile of instruction word counts (× 1.3 token expansion) |
| Expected oracle ratio | ~ 0.80 | Verified in Section 2 |